In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
import polars as pl
from hydra.utils import to_absolute_path

# Import your existing target generators
from fof8_core.loader import FOF8Loader
from fof8_core.features import get_draft_class
from fof8_core.targets import get_peak_overall, get_career_vorp

In [ ]:
OVERRIDE_CSV_PATH = "./stage1_oof_results.csv"
OVERRIDE_THRESHOLD = 0.24
OVERRIDE_PATH_FROM_MLFLOW = True
EXPERIMENT_NAME = "Economic_Talent_Engine"
LEAGUE_NAME = "DRAFT003"

df_oof = None

if OVERRIDE_CSV_PATH:
    csv_path = OVERRIDE_CSV_PATH
    if OVERRIDE_PATH_FROM_MLFLOW:
        df_oof = (
        pl.read_csv(csv_path)
        .rename({
            "y_true": "Target_Sieve",
            "cleared_sieve": "Pred_Sieve",
            "oof_prob": "P_Sieve"
            })
        )
    else:
        df_oof = (
            pl.read_csv(csv_path)
        )
        df_oof = df_oof.with_columns(
        pl.when(pl.col("P_Sieve") >= OVERRIDE_THRESHOLD).then(1).otherwise(0).alias("Pred_Sieve")
        )

else:
    mlflow.set_tracking_uri(
        
    )
    client = MlflowClient()
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

    # 1a. Get the latest parent pipeline run
    parent_runs = client.search_runs(
        experiment_ids=[experiment.experiment_id], 
        filter_string="tags.mlflow.runName LIKE 'Pipeline_%'",
        order_by=["start_time DESC"],
        max_results=1
    )
    parent_run_id = parent_runs[0].info.run_id
    # 1b. Get the child run that contains the actual results
    child_runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.parentRunId = '{parent_run_id}' AND tags.mlflow.runName = 'Stage1_Sieve_Classifier'",
        max_results=1
    )
    # 1c. Use the child's ID for the download
    latest_run_id = child_runs[0].info.run_id

    artifact_path = mlflow.artifacts.download_artifacts(
        run_id=latest_run_id, 
        artifact_path="stage1_oof_results.csv" 
    )

    # Load into Polars and rename columns to match notebook logic
    df_oof = (
        pl.read_csv(csv_path)
        .rename({
            "y_true": "Target_Sieve",
            "cleared_sieve": "Pred_Sieve",
            "oof_prob": "P_Sieve"
        })
    )

In [ ]:
# --- 2. Generate Ground Truth Targets ---
# Point to your raw data directory
loader = FOF8Loader(base_path="../fof8-gen/data/raw", league_name=LEAGUE_NAME)

print("Calculating ground truth targets...")
df_peak = get_peak_overall(loader)
df_vorp = get_career_vorp(loader)

# Combine into the continuous DGO target
df_targets = (
    df_peak.join(df_vorp, on="Player_ID", how="full")
    .fill_null(0) # Players who never played have 0 VORP/Peak
    .with_columns(
        (pl.col("Peak_Overall") * pl.col("Career_VORP")).alias("DGO")
    )
)

In [ ]:
# --- 3. Wire them together ---
df_evaluation = df_oof.join(df_targets, on="Player_ID", how="left")

# Fallback: If Target_Sieve is missing (e.g. loading a raw prediction file), 
# calculate it from the joined Peak/VORP data.
if "Target_Sieve" not in df_evaluation.columns:
    print("Target_Sieve missing from source. Calculating ground truth from raw data...")
    df_evaluation = df_evaluation.with_columns([
        pl.when((pl.col("Peak_Overall") >= 50) & (pl.col("Career_VORP") > 0.1))
        .then(1)
        .otherwise(0)
        .alias("Target_Sieve")
    ])

# Now calculate quadrants
df_evaluation = df_evaluation.with_columns([
    pl.when((pl.col("Target_Sieve") == 1) & (pl.col("Pred_Sieve") == 1)).then(pl.lit("1_True_Positive (Hit)"))
    .when((pl.col("Target_Sieve") == 1) & (pl.col("Pred_Sieve") == 0)).then(pl.lit("2_False_Negative (Missed Gem)"))
    .when((pl.col("Target_Sieve") == 0) & (pl.col("Pred_Sieve") == 1)).then(pl.lit("3_False_Positive (Drafted Bust)"))
    .when((pl.col("Target_Sieve") == 0) & (pl.col("Pred_Sieve") == 0)).then(pl.lit("4_True_Negative (Dodged Bust)"))
    .alias("Quadrant")
])


In [ ]:
# --- 4. The Economic Impact Report ---
print("\n=== Stage 1 Sieve Economic Impact Report ===")
report = (
    df_evaluation.group_by("Quadrant")
    .agg([
        pl.count("Player_ID").alias("Player_Count"),
        pl.col("Peak_Overall").mean().round(2).alias("Avg_Peak_Overall"),
        pl.col("Career_VORP").mean().round(3).alias("Avg_Career_VORP"),
        pl.col("DGO").mean().round(2).alias("Avg_DGO"),
        pl.col("DGO").sum().round(2).alias("Total_DGO_in_Cohort")
    ])
    .sort("Quadrant")
)

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(report.to_pandas())

In [ ]:
# --- 5. Deep Dive on the Missed Gems ---
false_negatives = df_evaluation.filter(pl.col("Quadrant") == "2_False_Negative (Missed Gem)")
if len(false_negatives) > 0:
    print(f"\n--- Top 5 Missed Gems (Highest DGO Thrown Away) ---")
    worst_misses = false_negatives.sort("DGO", descending=True).head(5)
    print(worst_misses.select(["Player_ID", "Year", "Peak_Overall", "Career_VORP", "DGO", "P_Sieve"]).to_pandas())

In [ ]:
missed_ids = [31785, 60234, 16823, 60236, 59358]
missed_years = [2055, 2089, 2037, 2089, 2088]

# Fetch the features for the specific years these players were drafted
dfs = []
for year in set(missed_years):
    df_class = get_draft_class(loader, year)
    dfs.append(df_class)

df_all_features = pl.concat(dfs)

# Isolate the missed gems
df_gems = df_all_features.filter(pl.col("Player_ID").is_in(missed_ids))

# Display the core traits that the model likely penalized
columns_to_inspect = [
    "Player_ID", "Position_Group", "Grade", 
    "Dash_Z", "Agility_Z", "Strength_Z", 
    "Developed"
]

# If you calculated Deltas for uncertainty, add a couple to see if they were highly uncertain
uncertainty_cols = [col for col in df_gems.columns if col.startswith("Delta_")][:3]

import pandas as pd
pd.set_option('display.max_columns', None)
print(df_gems.select(columns_to_inspect + uncertainty_cols).to_pandas())

In [ ]:
import polars as pl
import pandas as pd

# --- 1. Global Position Join (All Years) ---
# Clean up any existing position columns to avoid DuplicateErrors
cols_to_drop = [c for c in df_evaluation.columns if "Position_Group" in c]
if cols_to_drop:
    df_evaluation = df_evaluation.drop(cols_to_drop)

# Load the global draft data (all years)
df_all_draft = (
    loader.scan_file("rookies.csv")
    .select(["Player_ID", "Year", "Position_Group"])
    .collect()
)

# Perform the clean, global join
df_evaluation = (
    df_evaluation.join(
        df_all_draft, 
        on=["Player_ID", "Year"], 
        how="left"
    )
)

# --- 2. Define Custom Position Mapping ---
position_mapping = (
    pl.when(pl.col("Position_Group") == "QB").then(pl.lit("QB"))
    .when(pl.col("Position_Group").is_in(["SE", "FL", "WR"])).then(pl.lit("WR"))
    .when(pl.col("Position_Group").is_in(["LT", "RT", "LG", "RG", "C", "T", "G"])).then(pl.lit("OL"))
    .when(pl.col("Position_Group") == "TE").then(pl.lit("TE"))
    .when(pl.col("Position_Group") == "RB").then(pl.lit("RB"))
    .when(pl.col("Position_Group") == "FB").then(pl.lit("FB"))
    .when(pl.col("Position_Group").is_in(["LDE", "RDE", "DE"])).then(pl.lit("DE"))
    .when(pl.col("Position_Group").is_in(["LDT", "RDT", "NT", "DT"])).then(pl.lit("DT"))
    .when(pl.col("Position_Group").is_in(["SLB", "WLB", "MLB", "SILB", "WILB", "LB", "OLB", "ILB"])).then(pl.lit("LB"))
    .when(pl.col("Position_Group").is_in(["LCB", "RCB", "SS", "FS", "CB", "S"])).then(pl.lit("DB"))
    .when(pl.col("Position_Group").is_in(["K", "P", "LS"])).then(pl.lit("Special"))
    .otherwise(pl.lit("Other"))
    .alias("Custom_Position")
)

# Apply mapping
df_evaluation = df_evaluation.with_columns(position_mapping)

# --- 3. Generate the Grouped Report ---
position_report = (
    df_evaluation.group_by(["Custom_Position", "Quadrant"])
    .agg([
        pl.count("Player_ID").alias("Player_Count"),
        pl.col("Peak_Overall").mean().round(2).alias("Avg_Peak_Overall"),
        pl.col("Career_VORP").mean().round(3).alias("Avg_Career_VORP"),
        pl.col("DGO").mean().round(2).alias("Avg_DGO"),
        pl.col("DGO").sum().round(2).alias("Total_DGO")
    ])
    .sort(["Custom_Position", "Quadrant"])
)

# --- 4. Pivot: Total Value (DGO) Captured vs Lost ---
print("\n=== Total Value (DGO) Captured vs Lost by Position Group (2021-2100) ===")
pivot_dgo = position_report.pivot(
    index="Custom_Position",
    on="Quadrant",
    values="Total_DGO",
    aggregate_function="first"
).fill_null(0.0)

# Calculate the Value Retention Rate (How much of the total possible DGO did the model 'keep'?)
cols = pivot_dgo.columns
if "1_True_Positive (Hit)" in cols and "2_False_Negative (Missed Gem)" in cols:
    pivot_dgo = pivot_dgo.with_columns(
        (pl.col("1_True_Positive (Hit)") / (pl.col("1_True_Positive (Hit)") + pl.col("2_False_Negative (Missed Gem)"))).round(4).alias("Value_Retention_Rate")
    )

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(pivot_dgo.to_pandas().sort_values(by="Value_Retention_Rate", ascending=False) if "Value_Retention_Rate" in pivot_dgo.columns else pivot_dgo.to_pandas())

# --- 5. Pivot: Player Count Flow ---
print("\n=== Player Count Flow by Position Group (2021-2100) ===")
pivot_counts = position_report.pivot(
    index="Custom_Position",
    on="Quadrant",
    values="Player_Count",
    aggregate_function="first"
).fill_null(0)

print(pivot_counts.to_pandas())


In [ ]:
# Assuming df_evaluation is the dataframe from your economic report
magnitude_report = (
    df_evaluation.filter(pl.col("Quadrant") == "1_True_Positive (Hit)")
    .group_by("Custom_Position")
    .agg([
        pl.col("DGO").mean().round(2).alias("Average_Hit_DGO"),
        pl.col("DGO").quantile(0.90).round(2).alias("Top_10_Percent_DGO"),
        pl.col("DGO").max().round(2).alias("Absolute_Max_DGO")
    ])
    .sort("Top_10_Percent_DGO", descending=True)
)

import pandas as pd
print("\n=== The Magnitude of a Hit by Position ===")
print(magnitude_report.to_pandas())

In [ ]:
# Fetch personal ratings and game starts
lf_personal = loader.scan_file("players_personal.csv")
lf_records = loader.scan_file("player_record.csv")

clipboard_qbs = (
    lf_personal.select(["Player_ID", "Year", "Current_Overall"])
    .join(
        lf_records.select(["Player_ID", "Year", "Position_Group", "Experience", "S_Games_Started"]),
        on=["Player_ID", "Year"]
    )
    .filter(pl.col("Position_Group") == "QB")
    .group_by("Player_ID")
    .agg([
        # Total starts in their first 3 years
        pl.col("S_Games_Started").filter(pl.col("Experience") <= 2).sum().alias("Starts_Years_1_to_3"),
        # The peak rating they eventually hit in their career
        pl.col("Current_Overall").max().alias("Career_Peak_Overall")
    ])
    # Filter for QBs who sat early but became great
    .filter((pl.col("Starts_Years_1_to_3") <= 10) & (pl.col("Career_Peak_Overall") >= 60))
    .collect()
)

print(f"\nNumber of 'Clipboard' QBs who reached 60+ Overall: {len(clipboard_qbs)}")

In [ ]:
# 1. Fetch Salary Caps
lf_caps = loader.scan_file("universe_info.csv")
df_caps = (
    lf_caps.filter(pl.col("Information") == "Salary Cap (in tens of thousands)")
    .select(["Year", pl.col("Value/Round/Position").cast(pl.Int32).alias("Cap_10k")])
)

# 2. Build df_records (Loading fresh from source to avoid cached grouping)
lf_records = loader.scan_file("player_record.csv")
df_records = (
    lf_records.select([
        "Player_ID", "Year", 
        "Position",       # Granular (LDE, SE, etc.)
        "Position_Group", # Grouped (DE, WR, etc.)
        "Salary_Year_1", "Bonus_Year_1",
    ])
    .join(df_caps.lazy(), on="Year", how="left")
    .with_columns([
        ((pl.col("Salary_Year_1") + pl.col("Bonus_Year_1")) / pl.col("Cap_10k")).alias("Actual_Cap_Share")
    ])
).collect()

# 3. Helper function for the analysis
def calculate_scarcity_gap(df, group_col):
    return (
        df.filter(pl.col("Actual_Cap_Share") > 0)
        .with_columns(
            (pl.col("Actual_Cap_Share").rank(descending=True).over(["Year", group_col]) / 
             pl.len().over(["Year", group_col])).alias("Salary_Rank_Pct")
        )
        .group_by(group_col)
        .agg([
            pl.col("Actual_Cap_Share").filter(pl.col("Salary_Rank_Pct") <= 0.25).mean().alias("Starter_Rate_Cap_Pct"),
            pl.col("Actual_Cap_Share").filter((pl.col("Salary_Rank_Pct") > 0.25) & (pl.col("Salary_Rank_Pct") <= 0.50)).mean().alias("Backup_Rate_Cap_Pct")
        ])
        .with_columns(
            (pl.col("Starter_Rate_Cap_Pct") - pl.col("Backup_Rate_Cap_Pct")).alias("Positional_Premium")
        )
        .sort("Positional_Premium", descending=True)
    )

# 4. Generate and Print Both
df_market_granular = calculate_scarcity_gap(df_records, "Position")
df_market_groups = calculate_scarcity_gap(df_records, "Position_Group")

print("\n=== Market Rate Premium: Granular (LDE, RT, SE, etc.) ===")
print(df_market_granular.to_pandas())

print("\n=== Market Rate Premium: Position Groups (DE, T, WR, etc.) ===")
print(df_market_groups.to_pandas())
